In [1]:
import pandas  as  pd
! pip install pandasql
from  pandasql import sqldf

In [ ]:
users=pd.read_excel(r"...")
orders=pd.read_excel(r"...")
profiles=pd.read_excel(r"...")


In [ ]:
rides=pd.read_excel(r"...")
tags=pd.read_excel(r"...")

In [4]:
#Average order value per city
output=sqldf("""
select city,
        avg(price) as AOV
        from orders
        where status='FINISHED'
        group by city

           
 """)
output

,city,AOV
0,ISFAHAN,47946.619217
1,KARAJ,79757.425743
2,SHIRAZ,57678.260870
3,TEHRAN,65772.594752


In [5]:
#Number of order per city
output=sqldf("""
select city,
        count(*) Totalorders
        from orders
        where status='FINISHED'
        group by city

           
 """)
output

,city,Totalorders
0,ISFAHAN,281
1,KARAJ,202
2,SHIRAZ,115
3,TEHRAN,3087


In [6]:
#Number of order per day/per city
output=sqldf("""
select city,
        strftime("%Y-%m-%d", createdat) Day,
        count(*) Totoalorders
        from orders
        where status='FINISHED'
        group by city, day

           
 """)
output

,city,Day,Totoalorders
0,ISFAHAN,2023-05-08,8
1,ISFAHAN,2023-05-09,1
2,ISFAHAN,2023-05-10,19
3,ISFAHAN,2023-05-11,12
4,ISFAHAN,2023-05-12,20
...,...,...,...
97,TEHRAN,2023-06-02,58
98,TEHRAN,2023-06-03,142
99,TEHRAN,2023-06-04,29
100,TEHRAN,2023-06-05,50


In [7]:
#Number of order per week
output=sqldf("""
select city,
        strftime("%W", createdat) Week,
        count(*) Totoalorders
        from orders
        where status='FINISHED'
        group by city,  Week

           
 """)
output

,city,Week,Totoalorders
0,ISFAHAN,19,79
1,ISFAHAN,20,73
2,ISFAHAN,21,40
3,ISFAHAN,22,54
4,ISFAHAN,23,35
5,KARAJ,19,42
6,KARAJ,20,61
7,KARAJ,21,59
8,KARAJ,22,33
9,KARAJ,23,7


In [8]:
#Number of order per day of week 
output=sqldf("""
select 
        strftime('%w', createdat) Dayofweek,
        case when strftime("%w", createdat)='0' then 'Sunday'
         when strftime("%w", createdat)='1' then 'Monday'
         when strftime("%w", createdat)='2' then 'Tuesday'
         when strftime("%w", createdat)='3' then 'Wednesday'
         when strftime("%w", createdat)='4' then 'Thursday'
         when strftime("%w", createdat)='5' then 'Friday'
         when strftime("%w", createdat)='6' then 'Saturday'
        end as Dayofweek1,
        count (*) Totalorders
        from orders
        where status='FINISHED'
       group by Dayofweek1
       order by Totalorders desc
          
           
 """)
output


,Dayofweek,Dayofweek1,Totalorders
0,3,Wednesday,643
1,1,Monday,626
2,2,Tuesday,610
3,6,Saturday,596
4,4,Thursday,491
5,0,Sunday,479
6,5,Friday,240


In [9]:
#Fulfillment ratio (Finished/(Cancelation+Finished))
output=sqldf("""
select 
        count(case when status='FINISHED'then 1 end)*100/ COUNT(*) FFrate
        from orders       
 """)
output

,FFrate
0,77


In [10]:
#Fulfillment ratio per city (Finished/(Cancelation+Finished))
output=sqldf("""
select  city,
        count(case when status='FINISHED'then 1 end)*100/ COUNT(*) FFrate
        from orders 
        group by city      
 """)
output

,city,FFrate
0,ISFAHAN,81
1,KARAJ,69
2,SHIRAZ,76
3,TEHRAN,77


In [11]:
#Fulfillment ratio per city, per day of week  (Finished/(Cancelation+Finished))
output=sqldf("""
select  
        case when strftime("%w", createdat)='0' then 'Sunday'
        when strftime("%w", createdat)='1' then 'Monday'
        when strftime("%w", createdat)='2' then 'Tuesday'
        when strftime("%w", createdat)='3' then 'Wednesday'
        when strftime("%w", createdat)='4' then 'Thursday'
        when strftime("%w", createdat)='5' then 'Friday'
        when strftime("%w", createdat)='6' then 'Saturday'
        end as Dayofweek,
        count(case when status='FINISHED'then 1 end)*100/ COUNT(*) FFrate
        from orders 
        group by  Dayofweek    
 """)
output


,Dayofweek,FFrate
0,Friday,72
1,Monday,76
2,Saturday,77
3,Sunday,77
4,Thursday,78
5,Tuesday,78
6,Wednesday,78


In [12]:
#GMV,AOV, TotalOrder per city/category

output=sqldf("""
select  city, category,
        STRFTIME("%W", createdat) Weekofyear,
        AVG(price) AOV,
        count(id) Totoalorders,
        AVG(price)*count(id) GMV,
        AVG(price)*count(id)*.2 NMV
        from orders
        where status='FINISHED'
        group by city, category
 """)
output

,city,category,Weekofyear,AOV,Totoalorders,GMV,NMV
0,ISFAHAN,DELIVERY,21,67875.000000,8,543000.0,108600.0
1,ISFAHAN,NORMAL,19,47539.325843,267,12693000.0,2538600.0
2,ISFAHAN,VIP,19,39500.000000,6,237000.0,47400.0
3,KARAJ,DELIVERY,19,106800.000000,10,1068000.0,213600.0
4,KARAJ,NORMAL,19,78348.958333,192,15043000.0,3008600.0
5,SHIRAZ,DELIVERY,19,105333.333333,3,316000.0,63200.0
6,SHIRAZ,NORMAL,19,56459.459459,111,6267000.0,1253400.0
7,SHIRAZ,VIP,21,50000.000000,1,50000.0,10000.0
8,TEHRAN,DELIVERY,19,88924.528302,106,9426000.0,1885200.0
9,TEHRAN,NORMAL,19,65062.542488,2942,191414000.0,38282800.0


In [13]:
#Number of repeat customer (At least 2 purchase)
output=sqldf("""
select  userid,
        count(*) Totalorders
        from orders
        where status='FINISHED'
        group by userid
        having totalorders>1   
 """)
output

,userID,Totalorders
0,54731,2
1,649621,2
2,6890848,2
3,8414565,2
4,9978013,2
5,10786776,2
6,10932745,2
7,13447988,2
8,15803322,2
9,18030341,2


In [14]:
#Number of repeat customer with name (At least 2 purchase)
output=sqldf("""
with a as (select  userid,
        count(*) Totalorders
        from orders
        where status='FINISHED'
        group by userid
        having Totalorders>1 )
        
    select userid,firstName,lastName,Totalorders
      from a 
      left join users u on u.id=a.userid
      left join profiles p on p.id=u.profileid
 """)
output

,userid,firstName,lastName,Totalorders
0,54731,احمد,حبیبی,2
1,649621,طیبه,حسن زاده,2
2,6890848,مهدی,لک,2
3,8414565,اقا,جلالی علیایی,2
4,9978013,Amir,moradi,2
5,10786776,roya,خادم,2
6,10932745,Bahar,norouzi,2
7,13447988,آرزو,توکلی,2
8,15803322,marzieh,gholami,2
9,18030341,A.s,جمیلی,2


In [15]:
#Analysis and insight on Acquisition channels (CAC paid = 30KT)
#Overall CAC and Other insights

output=sqldf("""

   with a as(select 
                    city, userid,
                    count(*) totalorders
                from orders
                where status='FINISHED'
                group by city, userid
                )  

   select 
        city,
        count(*) Toatalacqusition,
        count(case when isorganic=1 then 1 end) organicaqusition,
        count(case when isorganic=0 then 1 end) paidaqusition, 
        count(case when a.totalorders>0 then 1 end) Firstpurchase,
        count(case when a.totalorders >0 and isorganic=1 then 1 end ) Firstpurchase_organic,
        count(case when isorganic=0 then 1 end)*30000 Totalcost,
        count(case when isorganic=0 then 1 end)*30000/count(*) CACOVERALL,
        count(case when isorganic=0 then 1 end)*30000/count(case when a.totalorders >0 then 1 end) CAC_Firstpurchase,
        count(case when isorganic=0 then 1 end)*30000/count(case when a.totalorders >0 and isorganic=0  then 1 end) CACpaid_Firstpurchase
      from users u
   left join profiles p on p.id=u.profileid
   left join a on u.id=a.userid  
   group by city
 """)
output

,city,Toatalacqusition,organicaqusition,paidaqusition,Firstpurchase,Firstpurchase_organic,Totalcost,CACOVERALL,CAC_Firstpurchase,CACpaid_Firstpurchase
0,NaN,4136,3862,274,0,0,8220000,1987,NaN,NaN
1,ISFAHAN,281,200,81,281,200,2430000,8647,8647.0,30000.0
2,KARAJ,201,159,42,201,159,1260000,6268,6268.0,30000.0
3,SHIRAZ,115,84,31,115,84,930000,8086,8086.0,30000.0
4,TEHRAN,3077,2127,950,3077,2127,28500000,9262,9262.0,30000.0


In [16]:
#avg time registration to purchase
output=sqldf("""
with a as (select 
        userid,
        min(createdat) firstpurchase
    from orders
    where status='FINISHED'
    group by userid)

select AVG(strftime("%J",firstpurchase) - strftime("%J",U.createdat)) AVGDAY
from a
left join users u on u.id=a.userid
      
 """)
output

,AVGDAY
0,1364.213896


In [17]:
#Cohort analysis
output=sqldf("""
with a as (select 
                     senderid, 
                     min (createdat) firstride
                from rides
                where status='FINISHED'
                group by senderid)

select 
        count(*) Totalrides,
        count(distinct r.senderid) Activesender,
        strftime("%W", firstride) Weekof_firtstride,
        strftime("%W", createdat) Weekof_ride

from a
left join rides r on r.senderid=a.senderid
where status='FINISHED'
 group by   Weekof_firtstride,  Weekof_ride
 """)
output

,Totalrides,Activesender,Weekof_firtstride,Weekof_ride
0,10491,4675,21,21
1,6010,2369,21,22
2,6188,2285,21,23
3,7375,2474,21,24
4,2475,1401,21,25
5,2577,1747,22,22
6,1142,624,22,23
7,1269,631,22,24
8,396,279,22,25
9,1848,1292,23,23


In [18]:
#Marketing campaign assessment
output=sqldf("""
select  Istreatment,
        count(id) Totalrides,
        count(distinct r.senderid) Segmsntsize ,
        sum(discount) Cost
        from rides r
    left join tags t on t.senderid=r.senderid
     where status='FINISHED'
     group by  Istreatment      
 """)
output


,isTreatment,Totalrides,Segmsntsize,Cost
0,0,8339,1835,0
1,1,34903,7374,53589000


In [19]:
#RFM segmentation(Recency,Frequency,Monetary)

output=sqldf("""
with RFM as
 (select  senderid,
        count(id) Frequency,
        avg(price) Monetary,
        strftime("%J", current_date) - max(strftime("%J", createdat)) Recency
    from rides
    where status='FINISHED'
    group by senderid),


    b as (
        select senderid,
        case when Recency <10 then 'Active'
                     when Recency <20 then 'Dormant'
                 else 'churn'
                end as  Recencysegment,
        -----------------------------
        case when Frequency <2 then 'PartTime'
                else 'FullTime'
                end as FrequencySegment,
        ------------------------------
        case when Monetary <200000 then 'LowValue'
                        else 'HoghValue'
                        end as MonetarySegment
    
                        
    from RFM)

select count(*), Recencysegment,FrequencySegment,MonetarySegment
from b 
group by Recencysegment,FrequencySegment,MonetarySegment

 """)
output





,count(*),Recencysegment,FrequencySegment,MonetarySegment
0,8,churn,FullTime,HoghValue
1,6101,churn,FullTime,LowValue
2,12,churn,PartTime,HoghValue
3,3088,churn,PartTime,LowValue


In [20]:
#Fraud detection
output=sqldf("""
select  senderid,
        Courierid,
        count(id) Totalorders
    from rides
    where status='FINISHED'
    group by Senderid, Courierid
    having   Totalorders>1    
 """)
output

,SenderID,CourierID,Totalorders
0,9.879479e+06,21516991.968840003,2
1,9.880922e+06,22035007.686440002,2
2,9.884436e+06,9851393.84404,2
3,9.885358e+06,22465203.94316,2
4,9.885358e+06,5054497.7526,2
...,...,...,...
75,1.006463e+07,16431435.641440002,2
76,1.006671e+07,9826715.6458,2
77,1.007050e+07,58894983.165680006,2
78,1.007050e+07,59280459.71404,2


In [21]:
#In what hours should we spend more on couriers?
output=sqldf("""
select  
        strftime("%H", createdat) Hourofday,
        count(*) Totalrides,
        round(avg(case when status='FINISHED'then 1 else 0 end),2) FFrate
    from rides
    group by  Hourofday
       
 """)
output

,Hourofday,Totalrides,FFrate
0,00,1686,0.27
1,01,811,0.29
2,02,484,0.31
3,03,337,0.31
4,04,439,0.29
5,05,1217,0.42
6,06,2705,0.43
7,07,4986,0.44
8,08,6588,0.45
9,09,7158,0.44


In [22]:
#Which city has the most discounts compared to sales?
output=sqldf("""
select   
        City,
       round(sum(discount)*1.0/sum(price),2) DiscountToPrice
        
    from rides
    where status='FINISHED'
    group by  City
    order by discounttoprice desc
       
 """)
output

,city,DiscountToPrice
0,NEYSHABOUR,0.17
1,ARDEBIL,0.13
2,GORGAN,0.11
3,HAMEDAN,0.09
4,ARAK,0.09
5,RASHT,0.08
6,KERMANSHAH,0.08
7,QOM,0.06
8,TABRIZ,0.05
9,AHVAZ,0.04
